In [ ]:
# 6-18-2026

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import regionmask
from tqdm.auto import tqdm

In [2]:
zarr_path = "seasfire_pyromes_ecoregions.zarr"

In [3]:
ds = xr.open_zarr(zarr_path, consolidated=True)

In [4]:
# also get ecoregion_domains.zarr and map those domains onto fire events (ba>0). see code from getting_pyromes_ecoregions.ipynb

In [6]:
print(list(ds.data_vars))

['area', 'biomes', 'cams_co2fire', 'cams_frpfire', 'drought_code_max', 'drought_code_mean', 'ecoregion', 'fcci_ba', 'fcci_ba_valid_mask', 'fcci_fraction_of_burnable_area', 'fcci_fraction_of_observed_area', 'fcci_number_of_patches', 'fwi_max', 'fwi_mean', 'gwis_ba', 'gwis_ba_valid_mask', 'lai', 'lccs_class_1', 'lccs_class_2', 'lccs_class_3', 'lccs_class_4', 'lccs_class_6', 'lccs_class_7', 'lsm', 'lst_day', 'ndvi', 'pop_dens', 'pyrome', 'rel_hum', 'skt', 'ssr', 'ssrd', 'sst', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 't2m_max', 't2m_mean', 't2m_min', 'tp', 'vpd', 'ws10']


In [7]:
domains = xr.open_zarr("ecoregion_domains.zarr", consolidated=True)

In [8]:
print(domains)
print()
print("dims:", dict(domains.dims))
print("coords:", list(domains.coords))
print("data vars:", list(domains.data_vars))

<xarray.Dataset> Size: 2MB
Dimensions:    (latitude: 720, longitude: 1440)
Coordinates:
  * latitude   (latitude) float64 6kB 89.88 89.62 89.38 ... -89.38 -89.62 -89.88
  * longitude  (longitude) float64 12kB -179.9 -179.6 -179.4 ... 179.6 179.9
Data variables:
    domain_id  (latitude, longitude) int16 2MB dask.array<chunksize=(180, 720), meta=np.ndarray>

dims: {'latitude': 720, 'longitude': 1440}
coords: ['latitude', 'longitude']
data vars: ['domain_id']


C:\Users\Yash\AppData\Local\Temp\ipykernel_2136\17572173.py:3: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("dims:", dict(domains.dims))


In [9]:
print("domains lat range:", float(domains.latitude.min()), "to", float(domains.latitude.max()))
print("domains lon range:", float(domains.longitude.min()), "to", float(domains.longitude.max()))
print("seasfire lat range:", float(ds.latitude.min()), "to", float(ds.latitude.max()))
print("seasfire lon range:", float(ds.longitude.min()), "to", float(ds.longitude.max()))

domains lat range: -89.875 to 89.875
domains lon range: -179.875 to 179.875
seasfire lat range: -89.875 to 89.875
seasfire lon range: -179.875 to 179.875


In [11]:
print("domains lat step:", float(domains.latitude.diff("latitude").mean()))
print("seasfire lat step:", float(ds.latitude.diff("latitude").mean()))

domains lat step: -0.25
seasfire lat step: -0.25


In [12]:
domain_vals = domains["domain_id"].values

print("dtype:", domain_vals.dtype)
print("min:", domain_vals.min(), "max:", domain_vals.max())
print("unique values (sample):", np.unique(domain_vals)[:20])
print("total unique:", len(np.unique(domain_vals)))
print("null/nan count:", np.sum(np.isnan(domain_vals.astype(float))))

dtype: int16
min: -1 max: 49
unique values (sample): [-1  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
total unique: 51
null/nan count: 0


In [13]:
print("total cells:", domain_vals.size)
print("non-domain cells (id == -1):", np.sum(domain_vals == -1))
print("domain cells (id >= 0):", np.sum(domain_vals >= 0))
print("non-domain fraction:", np.sum(domain_vals == -1) / domain_vals.size)

total cells: 1036800
non-domain cells (id == -1): 801012
domain cells (id >= 0): 235788
non-domain fraction: 0.7725810185185186


In [14]:
features_to_exclude = ["fcci_ba_valid_mask", "fcci_fraction_of_observed_area", "fcci_fraction_of_burnable_area", "fcci_number_of_patches"] # using gwis, not fcci
features_to_include = [v for v in ds.data_vars if v not in features_to_exclude]

In [15]:
domain_id_arr = domains["domain_id"].values

In [16]:
domain_id_arr

array([[-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       ...,
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1],
       [-1, -1, -1, ..., -1, -1, -1]], shape=(720, 1440), dtype=int16)

In [17]:
fire_mask = (ds["gwis_ba"].notnull()) & (ds["gwis_ba"] > 0)
time_idx, lat_idx, lon_idx = np.where(fire_mask.values)

In [18]:
data_dict = {
    "time": ds.time.values[time_idx],
    "latitude": ds.latitude.values[lat_idx],
    "longitude": ds.longitude.values[lon_idx],
    "gwis_ba_target": ds["gwis_ba"].values[time_idx, lat_idx, lon_idx],
    "domain_id": domain_id_arr[lat_idx, lon_idx]  # 2d, same indexing as pyrome
}

In [19]:
for var in tqdm(features_to_include):
    dims = ds[var].dims
    var_data = ds[var].values

    if dims == ("time", "latitude", "longitude"):
        data_dict[var] = var_data[time_idx, lat_idx, lon_idx]
    elif dims == ("latitude", "longitude"):
        data_dict[var] = var_data[lat_idx, lon_idx]
    elif dims == ("time",):
        data_dict[var] = var_data[time_idx]

  0%|          | 0/39 [00:00<?, ?it/s]

In [20]:
df = pd.DataFrame(data_dict)

In [21]:
print("rows:", len(df))
print("domain_id value counts (top 10):")
print(df["domain_id"].value_counts().head(10))
print("rows with domain_id == -1 (ocean/unclassified):", (df["domain_id"] == -1).sum())

rows: 3590844
domain_id value counts (top 10):
domain_id
11    652761
0     571858
45    352594
37    285273
20    204657
4     187710
5     171427
22    138804
2     122265
13    112415
Name: count, dtype: int64
rows with domain_id == -1 (ocean/unclassified): 50689


In [ ]:
print("domain_id value counts (bottom 20):")
print(df["domain_id"].value_counts().tail(20)) # bottom 16 domains have <2k fires, cutting those out gives 

domain_id value counts (bottom 20):
domain_id
39    7804
33    6561
7     5218
30    2434
15    1936
17    1921
24    1589
31    1477
10    1468
14    1450
48     821
9      751
3      548
35     336
34     335
43     270
42     102
40      27
44       2
41       1
Name: count, dtype: int64


In [24]:
df.to_csv("fire_events_with_domains.csv", index=False)

In [ ]:
# the bottom 16 domains have <2k samples. i will temporarily discard those to create a transfer matrix where N=34. this makes N(N-1)~1.1k domain pairs
# training would be N=30 N(N-1)=870 domain pairs, and the 4 held out domains for testing.
# further test on some of the discarded domains (only keep as target domain) to see if encoder can identify data-rich regimes can transfer well into data-sparce ones
# add 'data sparcity' as a feature to a physically constrained encoder model (num fires (ba>0) divided by domain area)